In [ ]:
import os
import json
import re
import time
import random
import pandas as pd
from tqdm import tqdm
import tinker
from tinker_cookbook import model_info, tokenizer_utils
from tinker_cookbook.renderers import get_renderer, get_text_content
from tinker.types import SamplingParams

class Config:
    # ------------------------------------------------------------
    # USER CONFIGURATION PARAMETERS
    # ------------------------------------------------------------
    MODE = 2                # 1: Augment existing QA (Mode 1), 2: Generate from Wikipedia (Mode 2)
    CONCURRENT_JOBS = 20    # Number of parallel jobs to send to Tinker at once
    MAX_SAMPLES = 5000      # Target number of QA pairs to generate (Mode 2) or process (Mode 1)
    
    # Model configuration
    MODEL_NAME = "openai/gpt-oss-120b"
    RENDERER_NAME = "gpt_oss_low_reasoning"
    
    # Paths (Kaggle working directory relative files)
    INPUT_CSV_PATH = "qa_gt_1000.csv"
    WIKI_CSV_PATH = "bangla_wikipedia_top_5.csv"
    OUTPUT_CSV_PATH = "synthetic_bangla_dataset.csv"

# Ensure Tinker API Key is set
if "TINKER_API_KEY" not in os.environ or os.environ["TINKER_API_KEY"].strip() == "":
    # Warn the user if key is missing
    print("WARNING: TINKER_API_KEY is not set in environment. Please set it before running Tinker API calls.")

# Cost rates for openai/gpt-oss-120b (per million tokens)
PREFILL_RATE = 0.18  # $0.18 per 1M input tokens
SAMPLE_RATE = 0.44   # $0.44 per 1M output tokens

In [ ]:
# Prompts
SYSTEM_PROMPT_MODE1 = (
    "You are a factual verification assistant. You will be provided with a Bengali context, "
    "a question, and the correct answer. Your task is to generate a highly plausible but "
    "incorrect/hallucinated answer in Bengali. The incorrect answer must look extremely realistic, "
    "be grammatically correct, and belong to one of the following categories:\n"
    "1. Entity Swap: replacing names, places, or organizations with related ones from the context or general knowledge.\n"
    "2. Numeric Alteration: changing dates, years, quantities, or counts slightly.\n"
    "3. Contradiction: inverting the truth value of a factual claim in the context.\n"
    "4. Baseless Addition: adding realistic details not mentioned in the context.\n\n"
    "Output ONLY the incorrect answer in Bengali. Do not include any English words, explanations, or labels."
)

SYSTEM_PROMPT_MODE2 = (
    "You are a factual dataset generator. You will be provided with a Bengali context snippet.\n"
    "Your task is to generate exactly 5 distinct and diverse question-answer pairs based on the context.\n"
    "For each pair, you must generate:\n"
    "1. A factual question in Bengali.\n"
    "2. The correct answer in Bengali directly supported by the context.\n"
    "3. A highly plausible incorrect/hallucinated answer in Bengali. The incorrect answer should fall into one of these categories:\n"
    "   - Entity Swap (replacing names, places, etc.)\n"
    "   - Numeric Alteration (changing dates, counts, numbers slightly)\n"
    "   - Contradiction (inverting a fact)\n"
    "   - Baseless Addition (adding realistic details not mentioned in the context)\n\n"
    "Output MUST be a valid JSON array of objects with the keys: \"question\", \"correct_answer\", \"wrong_answer\".\n"
    "Do not include markdown code block formatting (like ```json ... ```) or any other text before/after the JSON."
)

In [ ]:
# Helper Functions
def clean_json_string(raw_str):
    """Robustly extract JSON array from model response, handling thinking/conversational wrappers."""
    # Find the JSON array block starting with [ and ending with ]
    match = re.search(r'\[.*\]', raw_str, re.DOTALL)
    if match:
        return match.group(0).strip()
    return raw_str.strip()

def setup_tinker_client():
    print(f"Initializing Tinker ServiceClient for model: {Config.MODEL_NAME}...")
    service_client = tinker.ServiceClient()
    
    tokenizer = tokenizer_utils.get_tokenizer(Config.MODEL_NAME)
    try:
        renderer = get_renderer(Config.RENDERER_NAME, tokenizer)
        print(f"Using low-reasoning renderer: {Config.RENDERER_NAME}")
    except Exception as e:
        recommended = model_info.get_recommended_renderer_name(Config.MODEL_NAME)
        print(f"Renderer {Config.RENDERER_NAME} not found. Falling back to recommended: {recommended}")
        renderer = get_renderer(recommended, tokenizer)
        
    sampling_client = service_client.create_sampling_client(base_model=Config.MODEL_NAME)
    return tokenizer, renderer, sampling_client

In [ ]:
# Mode 1 Execution
def run_mode_1(tokenizer, renderer, sampling_client):
    print(f"Starting Mode 1: Augmenting existing QA from {Config.INPUT_CSV_PATH}")
    if not os.path.exists(Config.INPUT_CSV_PATH):
        raise FileNotFoundError(f"Input file not found at: {Config.INPUT_CSV_PATH}")
        
    df = pd.read_csv(Config.INPUT_CSV_PATH)
    print(f"Loaded {len(df)} source QA rows.")
    
    # Resume capability: check what is already processed
    processed_ids = set()
    if os.path.exists(Config.OUTPUT_CSV_PATH):
        try:
            processed_df = pd.read_csv(Config.OUTPUT_CSV_PATH)
            if "id" in processed_df.columns:
                processed_ids = set(processed_df["id"].dropna().astype(str).tolist())
                # Filter out already processed rows
                df = df[~df["id"].astype(str).isin(processed_ids)].reset_index(drop=True)
                print(f"Resuming: Loaded {len(processed_ids)} already processed rows. {len(df)} remaining to process.")
        except Exception as e:
            print(f"Warning: Failed to load existing output CSV for resuming: {e}")
            
    # Limit samples if configured
    if Config.MAX_SAMPLES > 0 and len(df) > Config.MAX_SAMPLES:
        df = df.head(Config.MAX_SAMPLES)
        print(f"Limited run to {len(df)} samples.")
        
    if len(df) == 0:
        print("All rows already processed. Nothing to do!")
        return
        
    stop_sequences = renderer.get_stop_sequences()
    params = SamplingParams(max_tokens=128, temperature=0.7, stop=stop_sequences)
    
    augmented_rows = []
    total_input_tokens = 0
    total_output_tokens = 0
    
    # Process in concurrent batches
    for i in tqdm(range(0, len(df), Config.CONCURRENT_JOBS), desc="Augmenting QA Batches"):
        batch = df.iloc[i : i + Config.CONCURRENT_JOBS]
        
        futures = []
        batch_rows = []
        batch_input_token_counts = []
        
        for _, row in batch.iterrows():
            context = str(row.get("context", "")).strip()
            question = str(row.get("question", "")).strip()
            correct_answer = str(row.get("correct_answer", "")).strip()
            row_id = row.get("id", "")
            
            user_content = (
                f"Context: {context}\n"
                f"Question: {question}\n"
                f"Correct Answer: {correct_answer}\n"
                f"Wrong Answer:"
            )
            
            messages = [
                {"role": "system", "content": SYSTEM_PROMPT_MODE1},
                {"role": "user", "content": user_content}
            ]
            
            prompt = renderer.build_generation_prompt(messages)
            future = sampling_client.sample(prompt, sampling_params=params, num_samples=1)
            futures.append(future)
            
            # Estimate input tokens
            input_tok_len = len(tokenizer.encode(str(prompt)))
            batch_input_token_counts.append(input_tok_len)
            
            batch_rows.append({
                "id": row_id,
                "context": context,
                "question": question,
                "correct_answer": correct_answer
            })
            
        batch_new_augmented = []
        
        # Collect and parse batch futures
        for idx, future in enumerate(futures):
            try:
                output = future.result()
                response_msg = renderer.parse_response(output.sequences[0].tokens)
                if isinstance(response_msg, tuple):
                    response_msg = response_msg[0]
                pred_text = get_text_content(response_msg).strip()
                
                # Track tokens
                out_tok_len = len(output.sequences[0].tokens)
                total_input_tokens += batch_input_token_counts[idx]
                total_output_tokens += out_tok_len
                
                row_data = batch_rows[idx]
                row_data["wrong_answer"] = pred_text
                batch_new_augmented.append(row_data)
            except Exception as e:
                print(f"Error processing row index {i + idx}: {e}")
                
        # Append batch directly to CSV
        if batch_new_augmented:
            batch_df = pd.DataFrame(batch_new_augmented)
            header_needed = not os.path.exists(Config.OUTPUT_CSV_PATH)
            batch_df.to_csv(Config.OUTPUT_CSV_PATH, mode='a', index=False, header=header_needed, encoding="utf-8")
            augmented_rows.extend(batch_new_augmented)
            
        # Calculate and print running cost
        running_cost = (total_input_tokens * PREFILL_RATE / 1e6) + (total_output_tokens * SAMPLE_RATE / 1e6)
        print(f"\rProcessed in this run: {len(augmented_rows)} | Est. Cost: ${running_cost:.4f} (In: {total_input_tokens}, Out: {total_output_tokens})", end="")
        
    final_cost = (total_input_tokens * PREFILL_RATE / 1e6) + (total_output_tokens * SAMPLE_RATE / 1e6)
    print(f"\nSuccessfully finished! Total rows in CSV: {len(processed_ids) + len(augmented_rows)}")
    print(f"Final Estimated API Cost (This Run): ${final_cost:.4f} (Input tokens: {total_input_tokens}, Output tokens: {total_output_tokens})")

In [ ]:
# Mode 2 Execution
def run_mode_2(tokenizer, renderer, sampling_client):
    print(f"Starting Mode 2: Generating synthetic QA from Wikipedia: {Config.WIKI_CSV_PATH}")
    if not os.path.exists(Config.WIKI_CSV_PATH):
        raise FileNotFoundError(f"Wikipedia CSV file not found at: {Config.WIKI_CSV_PATH}")
        
    wiki_df = pd.read_csv(Config.WIKI_CSV_PATH)
    print(f"Loaded {len(wiki_df)} Wikipedia articles.")
    
    # Deduplication and offset tracking structures
    offsets = {}           # article_id -> start_char_index
    depleted = set()       # set of article_ids that are fully consumed
    
    # Initialize offsets
    for _, row in wiki_df.iterrows():
        offsets[row["id"]] = 0
        
    # Resume capability: count already generated samples
    generated_count = 0
    if os.path.exists(Config.OUTPUT_CSV_PATH):
        try:
            existing_df = pd.read_csv(Config.OUTPUT_CSV_PATH)
            generated_count = len(existing_df)
            print(f"Resuming: Found {generated_count} existing QA pairs in {Config.OUTPUT_CSV_PATH}.")
        except Exception as e:
            print(f"Warning: Failed to load existing output CSV: {e}")
            
    if generated_count >= Config.MAX_SAMPLES:
        print(f"Target count of {Config.MAX_SAMPLES} already reached! Nothing to generate.")
        return
        
    generated_qa_pairs = []
    stop_sequences = renderer.get_stop_sequences()
    params = SamplingParams(max_tokens=1536, temperature=0.7, stop=stop_sequences)
    
    total_input_tokens = 0
    total_output_tokens = 0
    
    pbar = tqdm(initial=generated_count, total=Config.MAX_SAMPLES, desc="Generating QA Pairs")
    
    while generated_count < Config.MAX_SAMPLES:
        # 1. Filter articles that have not been depleted yet
        available_articles = wiki_df[~wiki_df["id"].isin(depleted)]
        if len(available_articles) == 0:
            print("All articles fully depleted! Stopping generation.")
            break
            
        # 2. Sample unique articles for this concurrent batch
        batch_size = min(Config.CONCURRENT_JOBS, len(available_articles))
        batch_df = available_articles.sample(n=batch_size, random_state=int(time.time()))
        
        futures = []
        batch_contexts = []
        batch_titles = []
        batch_input_token_counts = []
        
        for _, row in batch_df.iterrows():
            art_id = row["id"]
            content = str(row.get("content", "")).strip()
            title = str(row.get("title", "")).strip()
            start_idx = offsets[art_id]
            
            # Slicing chunk
            chunk = content[start_idx : start_idx + 2000]
            
            # Update offset
            offsets[art_id] = start_idx + 2000
            
            # Check depletion
            if len(content) - offsets[art_id] < 2000:
                depleted.add(art_id)
                
            if len(chunk) < 200: # Skip extremely short trailing chunks
                continue
                
            messages = [
                {"role": "system", "content": SYSTEM_PROMPT_MODE2},
                {"role": "user", "content": f"Context Snippet:\n{chunk}"}
            ]
            
            prompt = renderer.build_generation_prompt(messages)
            future = sampling_client.sample(prompt, sampling_params=params, num_samples=1)
            
            futures.append(future)
            batch_contexts.append(chunk)
            batch_titles.append(title)
            
            # Estimate input tokens
            input_tok_len = len(tokenizer.encode(str(prompt)))
            batch_input_token_counts.append(input_tok_len)
            
        if not futures:
            continue
            
        batch_new_pairs = []
        
        # Collect and parse batch futures
        for idx, future in enumerate(futures):
            try:
                output = future.result()
                response_msg = renderer.parse_response(output.sequences[0].tokens)
                if isinstance(response_msg, tuple):
                    response_msg = response_msg[0]
                pred_text = get_text_content(response_msg).strip()
                
                # Track tokens
                out_tok_len = len(output.sequences[0].tokens)
                total_input_tokens += batch_input_token_counts[idx]
                total_output_tokens += out_tok_len
                
                # Parse JSON array
                clean_json = clean_json_string(pred_text)
                try:
                    qa_list = json.loads(clean_json)
                    if isinstance(qa_list, list):
                        context = batch_contexts[idx]
                        title = batch_titles[idx]
                        valid_pairs = 0
                        for item in qa_list:
                            # Normalize keys to lowercase for robust extraction
                            normalized_item = {str(k).strip().lower(): v for k, v in item.items()}
                            
                            # Find question
                            q = None
                            for q_key in ["question", "prompt", "q"]:
                                if q_key in normalized_item:
                                    q = normalized_item[q_key]
                                    break
                            
                            # Find correct answer
                            c = None
                            for c_key in ["correct_answer", "correct", "answer", "correct answer", "a"]:
                                if c_key in normalized_item:
                                    c = normalized_item[c_key]
                                    break
                                    
                            # Find wrong answer
                            w = None
                            for w_key in ["wrong_answer", "wrong", "incorrect_answer", "wrong answer", "incorrect", "w"]:
                                if w_key in normalized_item:
                                    w = normalized_item[w_key]
                                    break
                                    
                            if q and c and w:
                                batch_new_pairs.append({
                                    "title": title,
                                    "context": context,
                                    "question": str(q).strip(),
                                    "correct_answer": str(c).strip(),
                                    "wrong_answer": str(w).strip()
                                })
                                valid_pairs += 1
                        
                        # Stop collecting if we reached total limit within this batch
                        if generated_count + len(batch_new_pairs) >= Config.MAX_SAMPLES:
                            break
                    else:
                        print("Warning: Parsed output is not a JSON list.")
                except json.JSONDecodeError:
                    print(f"JSON Decode Error. Raw response:\n{pred_text}\nCleaned:\n{clean_json}\n")
            except Exception as e:
                print(f"Error executing Tinker job: {e}")
                
        # Incremental save: append new batch pairs directly to CSV
        if batch_new_pairs:
            batch_df = pd.DataFrame(batch_new_pairs)
            header_needed = not os.path.exists(Config.OUTPUT_CSV_PATH)
            batch_df.to_csv(Config.OUTPUT_CSV_PATH, mode='a', index=False, header=header_needed, encoding="utf-8")
            generated_count += len(batch_new_pairs)
            generated_qa_pairs.extend(batch_new_pairs)
            pbar.update(len(batch_new_pairs))
            
        # Print running cost in tqdm description
        running_cost = (total_input_tokens * PREFILL_RATE / 1e6) + (total_output_tokens * SAMPLE_RATE / 1e6)
        pbar.set_postfix({"Est Cost": f"${running_cost:.4f}"})
            
    pbar.close()
    final_cost = (total_input_tokens * PREFILL_RATE / 1e6) + (total_output_tokens * SAMPLE_RATE / 1e6)
    print(f"Successfully generated and saved {len(generated_qa_pairs)} new QA pairs to {Config.OUTPUT_CSV_PATH}")
    print(f"Final Estimated API Cost (This Run): ${final_cost:.4f} (Input tokens: {total_input_tokens}, Output tokens: {total_output_tokens})")

In [ ]:
# Main Execution
if __name__ == "__main__":
    tokenizer, renderer, sampling_client = setup_tinker_client()
    
    # Create output directory if it doesn't exist
    out_dir = os.path.dirname(Config.OUTPUT_CSV_PATH)
    if out_dir:
        os.makedirs(out_dir, exist_ok=True)
        
    if Config.MODE == 1:
        run_mode_1(tokenizer, renderer, sampling_client)
    elif Config.MODE == 2:
        run_mode_2(tokenizer, renderer, sampling_client)
    else:
        print(f"Error: Unknown Config.MODE = {Config.MODE}")